In [29]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.utils.data as data
import torchvision.transforms as transforms
import torch.nn.functional as F
from pydicom import Dataset
from PIL import Image
from tqdm import tqdm

from src.utils.dataloader import BreastMNISTLoader, BCDRLoader, BUSBRALoader, INbreastLoader, BUSUCLMLoader, BreastLesionsUSGLoader
from src.models.main.models import (
    QuantumModel,
    ExtentedQuantumModel,
)
from src.utils.training import (
    training_loop,
    test_model,
)

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("CUDA is not available. Using CPU.")

Using: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 12.4


: 

In [ ]:
loader = BreastMNISTLoader(
            batch_size=16, 
            image_size=28,
        )

train_loader, test_loader, val_loader = loader.get_dataloaders()
dataloader = {"train": train_loader, "val": val_loader, "test": test_loader}

Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz


: 

In [ ]:
# loader = BreastLesionsUSGLoader(
#     batch_size=16,
#     image_size=224,
#     root_dir='src/data/BreastLesionsUSG/png'
# )

# train_loader, test_loader, val_loader = loader.get_dataloaders()

: 

: 

In [ ]:
# loader = BUSUCLMLoader(
#     batch_size=16,
#     image_size=28,
#     root_dir='src/data/BUS-UCLM/png'
# )

# train_loader, test_loader, val_loader = loader.get_dataloaders()

: 

: 

In [ ]:
# inbreast_loader = INbreastLoader(
#     batch_size=16,
#     image_size=28,
#     root_dir='src/data/INbreast/png'
# )

# train_loader, val_loader, test_loader = inbreast_loader.get_dataloaders()
# inbreast_dataloader = {"train": train_loader, "val": val_loader, "test": test_loader}

: 

: 

In [ ]:
# bcdr_loader = BCDRLoader(
#     batch_size=16,
#     image_size=28,
#     root_dir='src/data/BCDR/png'
# )

# train_loader, test_loader, val_loader = bcdr_loader.get_dataloaders()
# bcdr_dataloader = {"train": train_loader, "val": val_loader, "test": test_loader}

: 

: 

In [ ]:
n_qubits = 4
n_layers = 1
rand_params = np.random.uniform(high=2 * np.pi, size=(n_layers, n_qubits))
num_classes = 2
patch_size = 2

# Infer image size dynamically from loader to keep the model flexible.
sample_images, _ = next(iter(train_loader))
input_h, input_w = sample_images.shape[-2], sample_images.shape[-1]
output_size = (max(1, input_h // patch_size), max(1, input_w // patch_size))

print(f"Input size: {input_h}x{input_w} | Quantum output_size: {output_size}")

Input size: 28x28 | Quantum output_size: (14, 14)


: 

In [ ]:
model_basic = QuantumModel(
    n_qubits=n_qubits,
    n_layers=n_layers,
    rand_params=rand_params,
    output_size=output_size,
    patch_size=patch_size,
    num_classes=num_classes
).to(device)

print(f"QuantumModel created with {sum(p.numel() for p in model_basic.parameters())} parameters")

QuantumModel created with 0 parameters


: 

In [ ]:
model_trainable = ExtentedQuantumModel(
    n_qubits=n_qubits,
    n_layers=n_layers,
    num_classes=num_classes,
    patch_size=patch_size,
    mode="2d",
    stride=None,
    use_parallel=True,
).to(device)

print(f"ExtentedQuantumModel created")

Quantum layer has 4 trainable parameters
Requires grad: True
ExtentedQuantumModel created


: 

## Select Model for Training

Choose which model to use by uncommenting one of the lines below:

In [30]:
# Select one model for training:
model = model_basic  # Non-trainable quantum without encoder
# model = model_trainable  # Trainable quantum without encoder

print(f"\nSelected model: {model.__class__.__name__}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")


Selected model: QuantumModel
Total parameters: 1570
Trainable parameters: 1570


## Training Configuration

In [31]:
# Training hyperparameters
num_epochs = 20
learning_rate = 3e-3
save_dir = "extra"
os.makedirs(save_dir, exist_ok=True)

# Initialize FC layer with dummy forward pass (for non-trainable models)
if isinstance(model, (QuantumModel)):
    with torch.no_grad():
        dummy_input = torch.randn(1, 1, input_h, input_w).to(device)
        _ = model(dummy_input)
    print("✓ Model FC layer initialized")

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

print(f"\nTraining configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Loss function: CrossEntropyLoss")
print(f"  Save directory: {save_dir}")

✓ Model FC layer initialized

Training configuration:
  Epochs: 20
  Learning rate: 0.003
  Loss function: CrossEntropyLoss
  Save directory: extra


In [32]:
history = training_loop(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    save_dir=save_dir,
    save_best_only=True,
    save_all_epochs=True,
)

print("\n✓ Training completed!")

Epoch 1/20



Epoch 1 Summary:
  Train - Loss: 0.5609, Acc: 0.7143
  Val   - Loss: 0.5668, Acc: 0.7051, Precision: 0.9250, Recall: 0.6491, F1: 0.7629, AUC: 0.8329
  ✓ New best model saved (val_acc: 0.7051) at extra/best_model.pth
  ✓ Quantum rand_params sidecar saved at extra/best_model_rand_params.npy
Checkpoint saved to extra/checkpoints/model_epoch1.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch1_rand_params.npy
  Checkpoint saved at epoch 1
Epoch 2/20



Epoch 2 Summary:
  Train - Loss: 0.5492, Acc: 0.7198
  Val   - Loss: 0.4669, Acc: 0.8333, Precision: 0.8929, Recall: 0.8772, F1: 0.8850, AUC: 0.8505
  ✓ New best model saved (val_acc: 0.8333) at extra/best_model.pth
  ✓ Quantum rand_params sidecar saved at extra/best_model_rand_params.npy
Checkpoint saved to extra/checkpoints/model_epoch2.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch2_rand_params.npy
  Checkpoint saved at epoch 2
Epoch 3/20



Epoch 3 Summary:
  Train - Loss: 0.6029, Acc: 0.6850
  Val   - Loss: 0.4978, Acc: 0.8077, Precision: 0.8750, Recall: 0.8596, F1: 0.8673, AUC: 0.8095
Checkpoint saved to extra/checkpoints/model_epoch3.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch3_rand_params.npy
  Checkpoint saved at epoch 3
Epoch 4/20



Epoch 4 Summary:
  Train - Loss: 0.5510, Acc: 0.7033
  Val   - Loss: 0.4896, Acc: 0.7949, Precision: 0.8727, Recall: 0.8421, F1: 0.8571, AUC: 0.8079
Checkpoint saved to extra/checkpoints/model_epoch4.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch4_rand_params.npy
  Checkpoint saved at epoch 4
Epoch 5/20


...


Epoch 16 Summary:
  Train - Loss: 0.4695, Acc: 0.7619
  Val   - Loss: 0.4725, Acc: 0.7949, Precision: 0.8361, Recall: 0.8947, F1: 0.8644, AUC: 0.8037
Checkpoint saved to extra/checkpoints/model_epoch16.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch16_rand_params.npy
  Checkpoint saved at epoch 16
Epoch 17/20



Epoch 17 Summary:
  Train - Loss: 0.5068, Acc: 0.7326
  Val   - Loss: 0.4758, Acc: 0.7564, Precision: 0.8167, Recall: 0.8596, F1: 0.8376, AUC: 0.7970
Checkpoint saved to extra/checkpoints/model_epoch17.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch17_rand_params.npy
  Checkpoint saved at epoch 17
Epoch 18/20



Epoch 18 Summary:
  Train - Loss: 0.4741, Acc: 0.7491
  Val   - Loss: 0.5485, Acc: 0.7821, Precision: 0.8846, Recall: 0.8070, F1: 0.8440, AUC: 0.7870
Checkpoint saved to extra/checkpoints/model_epoch18.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch18_rand_params.npy
  Checkpoint saved at epoch 18
Epoch 19/20



Epoch 19 Summary:
  Train - Loss: 0.4551, Acc: 0.7784
  Val   - Loss: 0.5318, Acc: 0.7179, Precision: 0.8182, Recall: 0.7895, F1: 0.8036, AUC: 0.7694
Checkpoint saved to extra/checkpoints/model_epoch19.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch19_rand_params.npy
  Checkpoint saved at epoch 19
Epoch 20/20



Epoch 20 Summary:
  Train - Loss: 0.4640, Acc: 0.7729
  Val   - Loss: 0.5283, Acc: 0.7436, Precision: 0.8364, Recall: 0.8070, F1: 0.8214, AUC: 0.7828
Checkpoint saved to extra/checkpoints/model_epoch20.pth
Quantum rand_params sidecar saved to extra/checkpoints/model_epoch20_rand_params.npy
  Checkpoint saved at epoch 20

✓ Final model saved at extra/final_model.pth
✓ Quantum rand_params sidecar saved at extra/final_model_rand_params.npy

✓ Training completed!


In [ ]:
print("Initializing model layers...")
model.eval() 

best_model_path = os.path.join(save_dir, "best_model.pth")
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device, weights_only=True)

    # Support both formats:
    # 1) {"model_state_dict": ..., "epoch": ..., "val_acc": ...}
    # 2) raw state_dict (OrderedDict)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
        epoch = checkpoint.get("epoch", None)
        val_acc = checkpoint.get("val_acc", None)
    else:
        state_dict = checkpoint
        epoch = None
        val_acc = None

    model.load_state_dict(state_dict)
    print(f"✓ Loaded best model from {best_model_path}")
    print(f"  Epoch: {epoch if epoch is not None else 'N/A'}")
    print(f"  Val Accuracy: {val_acc:.4f}" if val_acc is not None else "  Val Accuracy: N/A")
else:
    print("⚠ Best model not found, using current weights")

hyperparams = {
    "epochs": num_epochs,
    "batch_size": 16,
    "learning_rate": learning_rate,
    "n_qubits": n_qubits,
    "n_layers": n_layers,
    "patch_size": patch_size,
    "model": model.__class__.__name__
}

test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = test_model(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
    hyperparams=hyperparams,
    output_file="noencoder.txt",
    extract_features=False,
    compute_detailed_metrics=True
)

print("\n" + "="*50)
print("FINAL TEST RESULTS")
print("="*50)
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall:    {test_rec:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print(f"Test AUC:       {test_auc:.4f}")
print("="*50)

Initializing model layers...
✓ Loaded best model from extra/best_model.pth
  Epoch: N/A
  Val Accuracy: N/A


Testing: 100%|██████████| 10/10 [00:30<00:00,  3.04s/it]

Test Loss: 0.5578
Accuracy: 0.7244, Precision: 0.8318, Recall: 0.7807, F1: 0.8054, AUC: 0.7776
Macro - Precision: 0.6608, Recall: 0.6761, F1: 0.6665
Micro - Precision: 0.7244, Recall: 0.7244, F1: 0.7244

FINAL TEST RESULTS
Test Loss:      0.5578
Test Accuracy:  0.7244
Test Precision: 0.8318
Test Recall:    0.7807
Test F1 Score:  0.8054
Test AUC:       0.7776


: 

In [ ]:
model.eval()
all_labels = []
all_predictions = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Generating predictions"):
        images = images.to(device)
        labels = labels.squeeze().to(device)
        
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        
        all_probs.extend(probs)
        all_predictions.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(f"\n✓ Visualizations complete!")

: 

: 